# 05_pure_tangent_stability: standalone resumable A100 campaign

redesigned G5 independent pure Born trajectories with online analytic occupied-empty tangent cocycle. This bundle uses its own versioned 25-trajectory output collection and never imports or pools legacy G4/G5 shards. The runtime check below must pass before any pilot or production work starts.

## 1. Mount Drive and inspect the immutable contract


In [ ]:
from pathlib import Path
import importlib.util, json, subprocess, sys
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
MYDRIVE = Path('/content/drive/MyDrive')
CAMPAIGN_ROOT = MYDRIVE / 'final_production_new_designs'
bundle_root = CAMPAIGN_ROOT / '05_pure_tangent_stability'
if not (bundle_root / 'run_bundle.py').is_file():
    raise FileNotFoundError(f'Missing uploaded bundle: {bundle_root}')
config = json.loads((bundle_root / 'production_config.json').read_text())
runner_path = bundle_root / 'run_bundle.py'
runner_spec = importlib.util.spec_from_file_location('_colab_05_pure_tangent_stability_runner', runner_path)
if runner_spec is None or runner_spec.loader is None:
    raise ImportError(f'Cannot load runner: {runner_path}')
runner_module = importlib.util.module_from_spec(runner_spec)
sys.modules[runner_spec.name] = runner_module
runner_spec.loader.exec_module(runner_module)
print(json.dumps(config, indent=2))


## 2. Verify the A100 runtime


In [ ]:
RUN_A100_PREFLIGHT = True
if RUN_A100_PREFLIGHT:
    subprocess.run([sys.executable, '-u', str(bundle_root / 'run_bundle.py'), 'preflight'], check=True)


## 3. Select and run the checksum-resumable queue


In [ ]:
RUN_MODE = 'production'  # or 'pilot'
CASE_ID = None           # e.g. 'G5_N20x20_wall'; None runs every case
SHARD_INDEX = None        # production: 0..4; None runs every shard
RUN_QUEUE = False
if RUN_MODE not in ('production', 'pilot'):
    raise ValueError("RUN_MODE must be 'production' or 'pilot'")
if RUN_MODE == 'pilot' and SHARD_INDEX is not None:
    raise ValueError('SHARD_INDEX is not used in pilot mode')
collection = (config['production_output_collection'] if RUN_MODE == 'production' else config['pilot_output_collection'])
output_root = MYDRIVE / collection
runner_argv = [RUN_MODE, '--output-root', str(output_root)]
if CASE_ID is not None:
    runner_argv += ['--case-id', CASE_ID]
if SHARD_INDEX is not None:
    runner_argv += ['--shard-index', str(SHARD_INDEX)]
command = [sys.executable, '-u', str(runner_path), *runner_argv]
print('[output root]', output_root)
print('[launch]', ' '.join(command))
if RUN_QUEUE:
    returncode = runner_module.main(runner_argv)
    if returncode not in (None, 0):
        raise RuntimeError(f'Runner returned nonzero status {returncode}')


## 4. Analyze only checksum-verified complete shards


In [ ]:
RUN_ANALYSIS = False
if RUN_ANALYSIS:
    subprocess.run([sys.executable, '-u', str(bundle_root / 'src' / 'g5_analysis.py'), '--archive-root', str(output_root), '--bundle-root', str(bundle_root)], check=True)


## Final. Disconnect this completed Colab runtime


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
